#Importing the dotenv and openai

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

#Fetching the documents

In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
print(f"Read {len(files)} files from the repository.")


    

# Understanding the Files

In [ ]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)
 # Print the first 500 characters of the content

In [ ]:
for doc in documents:
    print({
        "filename": doc["filename"],
        "chars": len(doc["content"])
    })

# No of pages in the document

In [ ]:
lesson_pages = [
    doc for doc in documents
    if "/lessons/" in doc["filename"]
]

print(f' The lesson pages in the documents are:',len(lesson_pages))

# Creating index

In [ ]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [ ]:
for doc in documents:
    print(doc["filename"])

In [ ]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py


In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
import importlib
import rag_helper

importlib.reload(rag_helper)

from rag_helper import RAGBase
load_dotenv()

openai_client = OpenAI()


filename = [doc["filename"] for doc in documents]

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    filename=filename,
)

answer, usage = assistant.rag("What is RAG?")

print("Answer:", answer)
print("Input tokens:", usage.input_tokens)
print("Output tokens:", usage.output_tokens)
print("Total tokens:", usage.total_tokens)

# Token Usage 

In [ ]:
answer, usage = assistant.rag("did it cover rag?")
print(answer)
print("Input tokens:", usage.input_tokens)
print("Output tokens:", usage.output_tokens)

# Chunking the documents into smaller pieces for better retrieval

In [ ]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Created {len(chunks)} chunks from the documents.")

# Minsearch index for chunks

In [ ]:
from minsearch import Index

chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

# assistant with chunked index

In [ ]:
answer, usage = assistant_chunks.rag("did it cover rag?")

print(answer)
print("Input tokens:", usage.input_tokens)
print("Output tokens:", usage.output_tokens)

# search tool definition with chunk_index

In [ ]:
def search( query, num_results=5):
        boost_dict = {
            "filename": 3.0,
            "content": 0.5,
        }
        if filename:
            filter_dict = {"filename": filename}
        else:
            filter_dict = None

       

        return chunk_index.search(
            query=query,
            num_results=num_results,
            boost_dict=boost_dict,
            filter_dict=filter_dict,
        )

# Function to handle the search tool call

In [ ]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the Filename for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the Filename and content fields of the documents.",
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

# Instructions for the agent to use the search tool

In [ ]:
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
""".strip()



# Function to handle the search tool call

In [ ]:
import json
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

# Full agentic loop implementation

In [ ]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

# agent loop using the search tool to answer a question

In [ ]:
agent_loop(instructions, "How do I run Olama locally?")


# Testing the agentic loop with a different question

In [ ]:
agent_loop(instructions, "How does the agentic loop work, and how is it different from plain RAG?")